# 🏍️ Motorcycle Spare Parts — ML Inventory Prediction System

**Business Problem:** Stock-outs due to no systematic ordering. Overseas lead time = 2–4 months.

---

## 📌 4 ML Models in This Notebook

| # | Model | Type | Goal |
|---|-------|------|------|
| 1 | **Demand Forecaster** | Time-Series (Moving Avg + Linear Trend) | Predict next month's quantity per part |
| 2 | **Reorder Classifier** | Random Forest Classification | Predict: should we reorder NOW or WAIT? |
| 3 | **Order Quantity Regressor** | Random Forest Regression | Predict: how many units to order? |
| 4 | **Stockout Risk Detector** | Rule-based + Scoring Model | Flag high-risk parts before they hit zero |

---
**Data Sources:**
- `Sales_Report_2025.xlsx` — 32,250 rows, 12 months, 3,579 unique parts
- `Jan_1st_week_Spare_Parts_PO.xlsx` — 1,629 PO rows including December loss orders

**⚠️ Note on Data Size:** Only 12 months of data available. ML models will use month-level aggregated features per part. For deeper accuracy, adding 2–3 more years of historical data is strongly recommended.

## 📦 Step 0: Install & Import Libraries

In [ ]:
# Core
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              mean_absolute_error, mean_squared_error, r2_score)
from sklearn.linear_model import LinearRegression

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# Style
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FFFFFF',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

MONTH_ORDER = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
MONTH_MAP   = {m: i+1 for i, m in enumerate(MONTH_ORDER)}

print('✅ All libraries loaded successfully')

## 📂 Step 1: Load & Explore Data

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
df_sales = pd.read_excel('Sales_Report_2025.xlsx')
df_po    = pd.read_excel('Jan_1st_week_Spare_Parts_PO.xlsx')

print('Sales shape :', df_sales.shape)
print('PO shape    :', df_po.shape)
df_sales.head(3)

In [ ]:
# ── EDA: Monthly sales trend + top parts ─────────────────────────────────────
df_pos = df_sales[df_sales['Sum of SlsVolQty'] > 0].copy()
df_pos['Month_Num'] = df_pos['Month'].map(MONTH_MAP)

monthly = df_pos.groupby('Month').agg(
    Qty     = ('Sum of SlsVolQty','sum'),
    Revenue = ('Sum of Net Sales_1','sum'),
    Parts   = ('Material','nunique')
).reindex(MONTH_ORDER)

top30 = (df_pos.groupby(['Material','Description'])['Sum of SlsVolQty']
         .sum().sort_values(ascending=False).head(30))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('📊 Sales Overview 2025', fontsize=15, fontweight='bold', y=1.02)

# Monthly qty
bars = axes[0].bar(monthly.index, monthly['Qty'], color='#2E75B6', alpha=0.85, edgecolor='white')
axes[0].set_title('Monthly Units Sold', fontweight='bold')
axes[0].set_ylabel('Quantity')
axes[0].tick_params(axis='x', rotation=45)
for bar in bars:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

# Top 10 parts
top10 = top30.head(10)
labels = [d[:30]+'...' if len(d)>30 else d for d in top10.index.get_level_values('Description')]
axes[1].barh(labels[::-1], top10.values[::-1], color='#1F4E79', alpha=0.85)
axes[1].set_title('Top 10 Parts by Annual Demand', fontweight='bold')
axes[1].set_xlabel('Annual Quantity')

plt.tight_layout()
plt.savefig('fig_eda.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\n📌 Total units sold (positive): {int(df_pos["Sum of SlsVolQty"].sum()):,}')
print(f'📌 Unique parts: {df_pos["Material"].nunique():,}')
print(f'📌 Date range: {df_pos["Month"].unique()}')

## 🔧 Step 2: Feature Engineering

Since we have only 12 months, we build **part-level monthly features** as our ML dataset.

Each row = one part × one month. Features include:
- Rolling demand stats (mean, std, max of prior months)
- Trend direction (is demand growing or falling?)
- Seasonality indicator (month number)
- Loss order flag (was this part ever stocked out?)
- Color-sensitive flag
- Lead time pressure score

In [ ]:
# ── 1. Build monthly pivot per part ──────────────────────────────────────────
monthly_part = (df_pos
    .groupby(['Material','Description','Month'])['Sum of SlsVolQty']
    .sum().reset_index())
monthly_part['Month_Num'] = monthly_part['Month'].map(MONTH_MAP)
monthly_part = monthly_part.sort_values(['Material','Month_Num'])

# ── 2. Part-level aggregate stats ────────────────────────────────────────────
part_stats = monthly_part.groupby(['Material','Description'])['Sum of SlsVolQty'].agg(
    Avg_Monthly   = 'mean',
    Std_Monthly   = 'std',
    Max_Monthly   = 'max',
    Min_Monthly   = 'min',
    Total_Annual  = 'sum',
    Months_Active = 'count'
).reset_index()
part_stats['Std_Monthly']  = part_stats['Std_Monthly'].fillna(0)
part_stats['CV']           = part_stats['Std_Monthly'] / (part_stats['Avg_Monthly'] + 1e-9)  # Coefficient of Variation

# ── 3. Trend feature: slope of demand over months ────────────────────────────
def calc_trend(grp):
    if len(grp) < 3:
        return 0.0
    x = grp['Month_Num'].values.reshape(-1,1)
    y = grp['Sum of SlsVolQty'].values
    lr = LinearRegression().fit(x, y)
    return round(lr.coef_[0], 4)

trends = (monthly_part.groupby('Material')
          .apply(calc_trend, include_groups=False)
          .reset_index(name='Demand_Trend'))

# ── 4. Loss order flag ────────────────────────────────────────────────────────
loss_mats = set(
    df_po[df_po['Customer Reference'].str.contains('Loss', case=False, na=False)]['Material']
)
part_stats['Was_Stocked_Out'] = part_stats['Material'].isin(loss_mats).astype(int)

# ── 5. Color flag ─────────────────────────────────────────────────────────────
color_kw = 'black|red|blue|white|silver|green|metallic|painted|SMX|MBL|YB |colour|color'
part_stats['Is_Color_Part'] = part_stats['Description'].str.contains(
    color_kw, case=False, na=False).astype(int)

# ── 6. Safety stock & reorder metrics ────────────────────────────────────────
Z        = 1.65   # 95% service level
LT_AVG   = 3      # avg lead time months
LT_MAX   = 4

part_stats['Safety_Stock']       = (Z * part_stats['Std_Monthly'] * np.sqrt(LT_AVG)).round(0)
part_stats['Reorder_Point']      = (part_stats['Avg_Monthly'] * LT_AVG + part_stats['Safety_Stock']).round(0)
part_stats['Reorder_Point_Max']  = (part_stats['Avg_Monthly'] * LT_MAX + part_stats['Safety_Stock']).round(0)
part_stats['Recommended_Order']  = (part_stats['Avg_Monthly'] * LT_AVG * 1.5).round(0)

# ── 7. Merge trend ────────────────────────────────────────────────────────────
features_df = part_stats.merge(trends, on='Material', how='left')
features_df['Demand_Trend'] = features_df['Demand_Trend'].fillna(0)

print(f'✅ Feature matrix built: {features_df.shape[0]} parts × {features_df.shape[1]} features')
features_df[['Material','Description','Avg_Monthly','Std_Monthly','CV',
             'Demand_Trend','Was_Stocked_Out','Is_Color_Part',
             'Safety_Stock','Reorder_Point']].head(10)

---
## 🤖 MODEL 1: Demand Forecasting
### Predict next month's demand per part

**Approach:** Since we have only 12 months, we use:
- **3-month weighted moving average** (practical, robust with short data)
- **Linear Trend projection** (captures growth/decline)
- **Seasonal adjustment** (month-of-year multiplier)

For each part: `Forecast = Trend_Projection × Seasonal_Factor`

In [ ]:
# ── Build month-level pivot for forecasting ───────────────────────────────────
pivot = (monthly_part.pivot_table(
    index='Material', columns='Month', values='Sum of SlsVolQty', aggfunc='sum'
).reindex(columns=MONTH_ORDER).fillna(0))

# ── Seasonal factors from all parts combined ──────────────────────────────────
month_total    = pivot.sum(axis=0)
seasonal_index = month_total / month_total.mean()  # >1 = above avg month

# ── Forecast function ─────────────────────────────────────────────────────────
def forecast_next_month(row, next_month_num=1):
    vals = row.values
    active = vals[vals > 0]
    if len(active) == 0:
        return 0.0
    # 3-month weighted moving average (most recent = highest weight)
    last3   = vals[-3:]
    weights = np.array([1, 2, 3])[:len(last3)]
    wma     = np.average(last3, weights=weights)
    # Linear trend adjustment
    trend_part = features_df[features_df['Material'] == row.name]['Demand_Trend']
    trend_val  = float(trend_part.values[0]) if len(trend_part) > 0 else 0
    trend_adj  = wma + trend_val
    # Seasonal factor for next month
    next_month_name = MONTH_ORDER[next_month_num % 12]
    s_factor        = seasonal_index.get(next_month_name, 1.0)
    forecast        = max(0, trend_adj * s_factor)
    return round(forecast, 1)

# ── Apply to top 60 parts by volume ──────────────────────────────────────────
top60 = features_df.nlargest(60, 'Total_Annual')['Material'].tolist()
pivot_top60 = pivot.loc[pivot.index.isin(top60)]

forecast_df = pd.DataFrame({
    'Material'          : pivot_top60.index,
    'Forecast_Next_Month': [forecast_next_month(row) for _, row in pivot_top60.iterrows()]
})
forecast_df = forecast_df.merge(
    features_df[['Material','Description','Avg_Monthly','Demand_Trend','Total_Annual']],
    on='Material'
).sort_values('Forecast_Next_Month', ascending=False)

forecast_df['vs_Avg_Monthly_%'] = (
    (forecast_df['Forecast_Next_Month'] - forecast_df['Avg_Monthly'])
    / (forecast_df['Avg_Monthly'] + 1e-9) * 100
).round(1)

# ── Evaluate on holdout (Dec as test month) ──────────────────────────────────
# Train on Jan-Nov, test on Dec
train_months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov']
test_month   = 'Dec'
pivot_train  = pivot[train_months]

preds, actuals = [], []
for mat, row in pivot_train.loc[pivot_train.index.isin(top60)].iterrows():
    last3   = row.values[-3:]
    weights = np.array([1, 2, 3])
    pred    = max(0, np.average(last3, weights=weights))
    actual  = pivot.loc[mat, test_month] if mat in pivot.index else 0
    preds.append(pred)
    actuals.append(actual)

mae  = mean_absolute_error(actuals, preds)
rmse = np.sqrt(mean_squared_error(actuals, preds))
r2   = r2_score(actuals, preds)

print(f'\n📊 MODEL 1 — Demand Forecast Evaluation (Dec holdout):')
print(f'   MAE  = {mae:.1f} units  (avg error per part per month)')
print(f'   RMSE = {rmse:.1f} units')
print(f'   R²   = {r2:.3f}  (1.0 = perfect)')
print(f'\n🔮 Top 10 Forecasted Parts — Next Month:')
forecast_df[['Description','Avg_Monthly','Forecast_Next_Month','vs_Avg_Monthly_%','Demand_Trend']].head(10)

In [ ]:
# ── Plot: Forecast vs Actual for top 6 parts ─────────────────────────────────
top6_mats = forecast_df.head(6)['Material'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('📈 MODEL 1: Demand Forecast — Actual vs Predicted (Top 6 Parts)', 
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, mat in enumerate(top6_mats):
    ax      = axes[idx]
    row     = pivot.loc[mat] if mat in pivot.index else pd.Series(dtype=float)
    desc    = forecast_df[forecast_df['Material']==mat]['Description'].values[0][:30]
    fcast   = forecast_df[forecast_df['Material']==mat]['Forecast_Next_Month'].values[0]

    months_plot = MONTH_ORDER + ['Jan\n(Fcst)']
    vals_plot   = list(row.reindex(MONTH_ORDER).fillna(0).values) + [fcast]
    colors      = ['#2E75B6'] * 12 + ['#FF4500']

    ax.bar(months_plot, vals_plot, color=colors, alpha=0.85, edgecolor='white')
    ax.axvline(x=11.5, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.set_title(f'{desc}...', fontsize=9, fontweight='bold')
    ax.set_ylabel('Units')
    ax.tick_params(axis='x', rotation=45, labelsize=7)

    # Annotation on forecast bar
    ax.text(12, fcast + max(vals_plot)*0.02, f'🔮{fcast:.0f}',
            ha='center', va='bottom', fontsize=8, color='#FF4500', fontweight='bold')

blue_p  = mpatches.Patch(color='#2E75B6', label='Actual (2025)')
red_p   = mpatches.Patch(color='#FF4500', label='Forecast (Next Month)')
fig.legend(handles=[blue_p, red_p], loc='lower center', ncol=2, fontsize=11, frameon=True)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('fig_model1_forecast.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ Model 1 complete. Seasonal index:')
print(seasonal_index.round(3))

---
## 🤖 MODEL 2: Reorder Classifier
### Should we place an order RIGHT NOW for this part?

**Target:** Binary — `1 = Reorder Now`, `0 = Wait`

**Logic for labelling:**
A part needs reordering NOW if its demand in the next lead-time window (3 months)
is likely to exceed available stock. We simulate this from the data.

**Algorithm:** Random Forest Classifier (handles non-linearity, gives feature importance)

In [ ]:
# ── Build labelled dataset ─────────────────────────────────────────────────────
# For each part × month: label = 1 if next 3-month demand > current month demand × 2
# (proxy for "stock insufficient for lead time")

rows = []
for mat in pivot.index:
    for m_idx, month in enumerate(MONTH_ORDER[:-3]):
        curr_demand       = pivot.loc[mat, month]
        future_3mo        = pivot.loc[mat, MONTH_ORDER[m_idx+1:m_idx+4]].sum()
        reorder_needed    = 1 if future_3mo > curr_demand * 2.5 else 0

        part_row          = features_df[features_df['Material'] == mat]
        if len(part_row) == 0:
            continue
        pr = part_row.iloc[0]

        rows.append({
            'Material'       : mat,
            'Month_Num'      : m_idx + 1,
            'Curr_Demand'    : curr_demand,
            'Avg_Monthly'    : pr['Avg_Monthly'],
            'Std_Monthly'    : pr['Std_Monthly'],
            'CV'             : pr['CV'],
            'Max_Monthly'    : pr['Max_Monthly'],
            'Demand_Trend'   : pr['Demand_Trend'],
            'Months_Active'  : pr['Months_Active'],
            'Was_Stocked_Out': pr['Was_Stocked_Out'],
            'Is_Color_Part'  : pr['Is_Color_Part'],
            'Safety_Stock'   : pr['Safety_Stock'],
            'Reorder_Label'  : reorder_needed
        })

clf_df = pd.DataFrame(rows)
print(f'Classification dataset: {clf_df.shape[0]:,} rows')
print(f'Class distribution — Reorder: {clf_df["Reorder_Label"].sum():,} | Wait: {(clf_df["Reorder_Label"]==0).sum():,}')

# ── Train / Test split ────────────────────────────────────────────────────────
FEAT_COLS = ['Month_Num','Curr_Demand','Avg_Monthly','Std_Monthly','CV',
             'Max_Monthly','Demand_Trend','Months_Active','Was_Stocked_Out',
             'Is_Color_Part','Safety_Stock']
X = clf_df[FEAT_COLS]
y = clf_df['Reorder_Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_clf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)

y_pred    = rf_clf.predict(X_test)
y_proba   = rf_clf.predict_proba(X_test)[:, 1]
cv_scores = cross_val_score(rf_clf, X, y, cv=5, scoring='f1')

print(f'\n📊 MODEL 2 — Reorder Classifier Results:')
print(f'   5-Fold CV F1 Score: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Wait","Reorder Now"])}')

In [ ]:
# ── Feature Importance + Confusion Matrix ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🤖 MODEL 2: Reorder Classifier — Feature Importance & Confusion Matrix',
             fontsize=13, fontweight='bold')

# Feature importance
importances = pd.Series(rf_clf.feature_importances_, index=FEAT_COLS).sort_values(ascending=True)
colors_imp  = ['#FF4500' if i > importances.quantile(0.7) else '#2E75B6' for i in importances]
importances.plot(kind='barh', ax=axes[0], color=colors_imp, edgecolor='white')
axes[0].set_title('Feature Importance (higher = more useful)', fontweight='bold')
axes[0].set_xlabel('Importance Score')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
im = axes[1].imshow(cm, cmap='Blues', aspect='auto')
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['Wait','Reorder Now'], fontsize=11)
axes[1].set_yticklabels(['Wait','Reorder Now'], fontsize=11)
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Actual', fontsize=12)
axes[1].set_title('Confusion Matrix', fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                     fontsize=18, fontweight='bold',
                     color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('fig_model2_classifier.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Predict reorder decision for current top 30 parts ─────────────────────────
current_month_num = 12   # December as current
top30_mats = features_df.nlargest(30,'Total_Annual')
pred_input = top30_mats[FEAT_COLS[1:]].copy()
pred_input.insert(0,'Month_Num', current_month_num)
pred_input.columns = FEAT_COLS

top30_mats = top30_mats.copy()
top30_mats['Reorder_Decision']    = rf_clf.predict(pred_input)
top30_mats['Reorder_Probability'] = rf_clf.predict_proba(pred_input)[:,1].round(3)
top30_mats['Decision_Label']      = top30_mats['Reorder_Decision'].map(
    {1:'🔴 REORDER NOW', 0:'🟢 WAIT'})

print('\n📋 Reorder Decisions — Top 30 Parts:')
top30_mats[['Description','Avg_Monthly','Reorder_Probability','Decision_Label']].head(15)

---
## 🤖 MODEL 3: Order Quantity Regression
### How many units should we order?

**Target:** Continuous — `Recommended Order Quantity`

**Label = Avg_Monthly × Lead_Time_Months + Safety_Stock** (derived from data, not manual)

**Algorithm:** Gradient Boosting Regressor (best for tabular data with non-linear patterns)

In [ ]:
# ── Build regression dataset ──────────────────────────────────────────────────
reg_df = features_df[features_df['Avg_Monthly'] >= 1].copy()
reg_df['Order_Qty_Target'] = reg_df['Recommended_Order']   # = Avg × 3 × 1.5

REG_FEATS = ['Avg_Monthly','Std_Monthly','CV','Max_Monthly','Min_Monthly',
             'Demand_Trend','Months_Active','Was_Stocked_Out',
             'Is_Color_Part','Safety_Stock','Total_Annual']

X_reg  = reg_df[REG_FEATS]
y_reg  = reg_df['Order_Qty_Target']

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# ── Gradient Boosting Regressor ───────────────────────────────────────────────
gbr = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    min_samples_leaf=3, subsample=0.8, random_state=42)
gbr.fit(X_tr, y_tr)

y_pred_reg = gbr.predict(X_te)
mae_reg    = mean_absolute_error(y_te, y_pred_reg)
rmse_reg   = np.sqrt(mean_squared_error(y_te, y_pred_reg))
r2_reg     = r2_score(y_te, y_pred_reg)

print('📊 MODEL 3 — Order Quantity Regressor Results:')
print(f'   MAE  = {mae_reg:.1f} units   (avg prediction error)')
print(f'   RMSE = {rmse_reg:.1f} units')
print(f'   R²   = {r2_reg:.4f}   (1.0 = perfect fit)')

# ── Predict order quantities for all parts ────────────────────────────────────
reg_df = reg_df.copy()
reg_df['ML_Order_Qty']   = gbr.predict(X_reg).round(0).astype(int)
reg_df['Rule_Order_Qty'] = reg_df['Recommended_Order'].astype(int)
reg_df['Qty_Diff']       = reg_df['ML_Order_Qty'] - reg_df['Rule_Order_Qty']

print(f'\n🔮 Sample Order Quantity Predictions:')
reg_df.nlargest(15,'Avg_Monthly')[[
    'Description','Avg_Monthly','Safety_Stock',
    'Rule_Order_Qty','ML_Order_Qty','Qty_Diff']].head(15)

In [ ]:
# ── Plot: Actual vs Predicted + Residuals ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('🤖 MODEL 3: Order Quantity Regression — Diagnostics',
             fontsize=13, fontweight='bold')

# Actual vs Predicted scatter
ax = axes[0]
ax.scatter(y_te, y_pred_reg, alpha=0.5, color='#2E75B6', s=20)
lim = max(y_te.max(), max(y_pred_reg)) * 1.05
ax.plot([0, lim], [0, lim], 'r--', linewidth=2, label='Perfect Fit')
ax.set_xlabel('Actual Order Qty'); ax.set_ylabel('Predicted Order Qty')
ax.set_title(f'Actual vs Predicted\nR² = {r2_reg:.4f}', fontweight='bold')
ax.legend()

# Residuals histogram
residuals = y_te.values - y_pred_reg
axes[1].hist(residuals, bins=40, color='#2E75B6', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title(f'Residuals Distribution\nMAE={mae_reg:.1f}', fontweight='bold')
axes[1].set_xlabel('Error (Actual − Predicted)')

# Feature importance
imp_reg = pd.Series(gbr.feature_importances_, index=REG_FEATS).sort_values(ascending=True)
imp_reg.plot(kind='barh', ax=axes[2], color='#1F4E79', edgecolor='white', alpha=0.85)
axes[2].set_title('Feature Importance', fontweight='bold')
axes[2].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('fig_model3_regression.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🤖 MODEL 4: Stockout Risk Detector
### Flag parts that are likely to run out of stock before the next order arrives

**Approach:** Composite Risk Scoring Model

Each part gets a **Risk Score (0–100)** based on:

| Factor | Weight | Why |
|--------|--------|-----|
| High demand variability (CV) | 25% | Unpredictable = risky |
| Rising demand trend | 20% | Growing demand depletes stock faster |
| Previously stocked out | 20% | History repeats |
| Low months active | 15% | Sparse data = uncertain |
| High avg demand | 20% | High velocity = faster depletion |

**Output:** Risk tier — 🔴 CRITICAL / 🟠 HIGH / 🟡 MEDIUM / 🟢 LOW

In [ ]:
# ── Risk scoring ───────────────────────────────────────────────────────────────
risk_df = features_df.copy()

def minmax(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn + 1e-9)

# Normalize each risk factor 0→1
risk_df['r_cv']         = minmax(risk_df['CV'])                              # high CV = high risk
risk_df['r_trend']      = minmax(risk_df['Demand_Trend'].clip(lower=0))      # rising trend = risky
risk_df['r_stockout']   = risk_df['Was_Stocked_Out'].astype(float)           # already 0/1
risk_df['r_sparse']     = 1 - minmax(risk_df['Months_Active'])               # fewer months = riskier
risk_df['r_highdemand'] = minmax(risk_df['Avg_Monthly'])                     # high demand = risky

# Weighted composite score
risk_df['Risk_Score'] = (
    0.25 * risk_df['r_cv'] +
    0.20 * risk_df['r_trend'] +
    0.20 * risk_df['r_stockout'] +
    0.15 * risk_df['r_sparse'] +
    0.20 * risk_df['r_highdemand']
) * 100

risk_df['Risk_Score'] = risk_df['Risk_Score'].round(1)

# Risk tier
def risk_tier(score):
    if score >= 70: return '🔴 CRITICAL'
    elif score >= 50: return '🟠 HIGH'
    elif score >= 30: return '🟡 MEDIUM'
    else: return '🟢 LOW'

risk_df['Risk_Tier'] = risk_df['Risk_Score'].apply(risk_tier)

# Summary
tier_counts = risk_df['Risk_Tier'].value_counts()
print('📊 MODEL 4 — Stockout Risk Distribution:')
print(tier_counts.to_string())
print(f'\n⚠️  Parts requiring IMMEDIATE attention (CRITICAL + HIGH):')
print(f'   {(risk_df["Risk_Tier"].str.contains("CRITICAL|HIGH")).sum()} parts out of {len(risk_df)}')

print(f'\n🔴 TOP 20 CRITICAL/HIGH RISK PARTS:')
risk_df.sort_values('Risk_Score', ascending=False)[
    ['Description','Avg_Monthly','CV','Demand_Trend',
     'Was_Stocked_Out','Risk_Score','Risk_Tier']
].head(20)

In [ ]:
# ── Risk visualization ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle('🚨 MODEL 4: Stockout Risk Detector', fontsize=14, fontweight='bold')
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Risk distribution donut
ax1 = fig.add_subplot(gs[0, 0])
tier_order  = ['🔴 CRITICAL','🟠 HIGH','🟡 MEDIUM','🟢 LOW']
tier_colors = ['#C00000','#FF6600','#FFD700','#00B050']
counts = [tier_counts.get(t,0) for t in tier_order]
wedges, texts, autotexts = ax1.pie(
    counts, labels=tier_order, colors=tier_colors,
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2},
    pctdistance=0.75, textprops={'fontsize':9})
for at in autotexts: at.set_fontweight('bold')
ax1.set_title('Risk Tier Distribution', fontweight='bold')

# 2. Risk score histogram
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(risk_df['Risk_Score'], bins=30, color='#2E75B6', edgecolor='white', alpha=0.85)
ax2.axvline(70, color='#C00000', linestyle='--', linewidth=2, label='Critical (70)')
ax2.axvline(50, color='#FF6600', linestyle='--', linewidth=2, label='High (50)')
ax2.axvline(30, color='#FFD700', linestyle='--', linewidth=2, label='Medium (30)')
ax2.set_title('Risk Score Distribution', fontweight='bold')
ax2.set_xlabel('Risk Score (0–100)'); ax2.set_ylabel('# Parts')
ax2.legend(fontsize=8)

# 3. Top 15 critical parts bar
ax3 = fig.add_subplot(gs[0, 2])
top15_risk = risk_df.nlargest(15,'Risk_Score')
bar_colors = ['#C00000' if '🔴' in t else '#FF6600' for t in top15_risk['Risk_Tier']]
labels15   = [str(d)[:28]+'…' if len(str(d))>28 else str(d) for d in top15_risk['Description']]
ax3.barh(labels15[::-1], top15_risk['Risk_Score'].values[::-1],
         color=bar_colors[::-1], edgecolor='white', alpha=0.9)
ax3.set_title('Top 15 Highest Risk Parts', fontweight='bold')
ax3.set_xlabel('Risk Score')
ax3.tick_params(axis='y', labelsize=7)

# 4. Demand vs CV bubble chart
ax4 = fig.add_subplot(gs[1, :])
top_risk = risk_df.nlargest(40,'Risk_Score')
tier_color_map = {
    '🔴 CRITICAL':'#C00000','🟠 HIGH':'#FF6600',
    '🟡 MEDIUM':'#FFD700','🟢 LOW':'#00B050'}
scatter_colors = [tier_color_map[t] for t in top_risk['Risk_Tier']]
sc = ax4.scatter(
    top_risk['Avg_Monthly'], top_risk['CV'],
    s=top_risk['Risk_Score']*4, c=scatter_colors,
    alpha=0.75, edgecolors='white', linewidths=0.8)
ax4.set_title('Demand vs Variability Bubble Chart (bubble size = Risk Score) — Top 40 Risk Parts',
              fontweight='bold')
ax4.set_xlabel('Avg Monthly Demand')
ax4.set_ylabel('Coefficient of Variation (CV)')
ax4.axhline(1.0, color='gray', linestyle=':', alpha=0.5)

for _, row in top_risk.nlargest(10,'Risk_Score').iterrows():
    ax4.annotate(str(row['Description'])[:18], (row['Avg_Monthly'], row['CV']),
                 fontsize=6.5, ha='left', va='bottom',
                 xytext=(3,3), textcoords='offset points')

patches = [mpatches.Patch(color=c, label=t) for t,c in tier_color_map.items()]
ax4.legend(handles=patches, loc='upper right', fontsize=9, frameon=True)

plt.savefig('fig_model4_risk.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 📋 Step 3: Final Combined Output — Ordering Action Plan
### One table that combines all 4 models into actionable decisions per part

In [ ]:
# ── Merge all model outputs ───────────────────────────────────────────────────
action_plan = features_df.nlargest(60,'Total_Annual')[[
    'Material','Description','Avg_Monthly','Total_Annual',
    'Is_Color_Part','Was_Stocked_Out','Reorder_Point','Safety_Stock']].copy()

# Model 1: Forecast
action_plan = action_plan.merge(
    forecast_df[['Material','Forecast_Next_Month','vs_Avg_Monthly_%']],
    on='Material', how='left')

# Model 2: Classifier
action_plan = action_plan.merge(
    top30_mats[['Material','Reorder_Decision','Reorder_Probability','Decision_Label']],
    on='Material', how='left')
action_plan['Reorder_Decision'] = action_plan['Reorder_Decision'].fillna(0).astype(int)
action_plan['Decision_Label']   = action_plan['Decision_Label'].fillna('🟢 WAIT')
action_plan['Reorder_Probability'] = action_plan['Reorder_Probability'].fillna(0)

# Model 3: Order quantity
reg_top = reg_df[['Material','ML_Order_Qty','Rule_Order_Qty']]
action_plan = action_plan.merge(reg_top, on='Material', how='left')
action_plan['ML_Order_Qty']   = action_plan['ML_Order_Qty'].fillna(
    action_plan['Avg_Monthly'] * 3).astype(int)
action_plan['Rule_Order_Qty'] = action_plan['Rule_Order_Qty'].fillna(
    action_plan['Avg_Monthly'] * 3).astype(int)

# Model 4: Risk
action_plan = action_plan.merge(
    risk_df[['Material','Risk_Score','Risk_Tier']], on='Material', how='left')

# Final priority flag
def final_action(row):
    if row['Risk_Tier'] in ['🔴 CRITICAL'] or row['Reorder_Decision'] == 1:
        return '🔴 ORDER IMMEDIATELY'
    elif row['Risk_Tier'] in ['🟠 HIGH']:
        return '🟠 ORDER THIS WEEK'
    elif row['Risk_Tier'] in ['🟡 MEDIUM']:
        return '🟡 PLAN ORDER SOON'
    else:
        return '🟢 MONITOR'

action_plan['Final_Action'] = action_plan.apply(final_action, axis=1)

# Sort by urgency
urgency_order = {'🔴 ORDER IMMEDIATELY':0,'🟠 ORDER THIS WEEK':1,
                 '🟡 PLAN ORDER SOON':2,'🟢 MONITOR':3}
action_plan['_sort'] = action_plan['Final_Action'].map(urgency_order)
action_plan = action_plan.sort_values(['_sort','Risk_Score'], ascending=[True,False]).drop('_sort',axis=1)

print('\n📋 FINAL ORDERING ACTION PLAN — TOP 30 PARTS:')
display_cols = ['Description','Avg_Monthly','Forecast_Next_Month',
                'Safety_Stock','ML_Order_Qty','Risk_Score','Risk_Tier',
                'Reorder_Probability','Is_Color_Part','Final_Action']
action_plan[display_cols].head(30)

In [ ]:
# ── Final summary dashboard chart ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('📋 FINAL ACTION PLAN — Combined ML Output (Top 25 Parts)',
             fontsize=14, fontweight='bold')

top25 = action_plan.head(25)
labels25 = [str(d)[:32]+'…' if len(str(d))>32 else str(d) for d in top25['Description']]
action_colors = {
    '🔴 ORDER IMMEDIATELY':'#C00000',
    '🟠 ORDER THIS WEEK':'#FF6600',
    '🟡 PLAN ORDER SOON':'#FFD700',
    '🟢 MONITOR':'#00B050'
}
bar_c = [action_colors.get(a,'#2E75B6') for a in top25['Final_Action']]

# Risk score bar
axes[0].barh(labels25[::-1], top25['Risk_Score'].values[::-1],
             color=bar_c[::-1], edgecolor='white', alpha=0.9, height=0.7)
axes[0].set_title('Risk Score by Part (colored by action)', fontweight='bold')
axes[0].set_xlabel('Risk Score (0–100)')
axes[0].tick_params(axis='y', labelsize=8)
patches = [mpatches.Patch(color=c, label=l) for l,c in action_colors.items()]
axes[0].legend(handles=patches, loc='lower right', fontsize=8, frameon=True)

# ML Order Qty vs Avg Monthly
x = np.arange(len(top25))
w = 0.38
axes[1].bar(x-w/2, top25['Avg_Monthly'].values, width=w,
            label='Avg Monthly Demand', color='#2E75B6', alpha=0.85, edgecolor='white')
axes[1].bar(x+w/2, top25['ML_Order_Qty'].values, width=w,
            label='ML Recommended Order Qty', color='#FF4500', alpha=0.85, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels([l[:18] for l in labels25], rotation=75, fontsize=7)
axes[1].set_title('Avg Monthly Demand vs ML Order Quantity', fontweight='bold')
axes[1].set_ylabel('Units')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_final_action_plan.png', dpi=120, bbox_inches='tight')
plt.show()

# Save action plan to CSV
action_plan.to_csv('ML_Action_Plan_Output.csv', index=False)
print('\n✅ Action plan saved to: ML_Action_Plan_Output.csv')

---
## ✅ Summary & Next Steps

### What each model does for your business:

| Model | Output | Business Use |
|-------|--------|--------------|
| **1. Demand Forecast** | Predicted qty for next month | Know what to expect before stock runs out |
| **2. Reorder Classifier** | Yes/No — order now? | Never miss an order trigger |
| **3. Order Qty Regressor** | How many units to order | Avoid over/under-ordering overseas |
| **4. Stockout Risk Score** | 0–100 risk + tier | Prioritize urgent parts |

### ⚠️ Limitations with 1 Year of Data
- Models are **trained and tested on same year** — add more years for stronger validation
- **Seasonality** is estimated but not deeply modelled
- **Color split ratios** need a dedicated colour-tracking column in your sales data

### 🚀 Recommended Next Steps
1. **Collect 2–3 more years** of sales data → retrain all models for much higher accuracy
2. **Add stock-on-hand column** to your sales file → enables true inventory level tracking
3. **Add colour column** to sales data → enables colour-specific demand forecasting
4. **Automate** this notebook monthly using a scheduler (e.g., Windows Task Scheduler + `jupyter nbconvert`)
5. **Connect to Excel model** — use `ML_Action_Plan_Output.csv` as input to the Excel dashboard built previously
